# Continous Evidence

In [ ]:
from pathlib import Path

import hashlib
import numpy as np
import pandas as pd

METHOD_VERSION = "1.0.0"
EXPECTED_SIZE_BYTES = 70_025_339
EXPECTED_SHAPE = (188_318, 132)
EXPECTED_SHA256 = None  # Set only if the team has an approved reference hash.
DEFAULT_QUANTILE_BINS = 10
HISTOGRAM_BINS = 30
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
EXPECTED_HEADER = ["id"] + [f"cat{i}" for i in range(1, 117)] + CONT_COLS + ["loss"]


def locate_root():
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (folder / "data" / "allstate_claims_data.csv").is_file():
            return folder
    raise FileNotFoundError("Could not find data/allstate_claims_data.csv. Open the notebook within the repository.")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = locate_root()
SOURCE = ROOT / "data" / "allstate_claims_data.csv"
EVIDENCE_DIR = ROOT / "notebooks" / "final-deliverables" / "September" / "Gate 3" / "continuous_evidence"

source_size = SOURCE.stat().st_size
source_hash = sha256(SOURCE)
if source_size != EXPECTED_SIZE_BYTES:
    raise RuntimeError(f"Incorrect source size: {source_size:,} bytes; expected {EXPECTED_SIZE_BYTES:,}. Stop and investigate.")
if EXPECTED_SHA256 is not None and source_hash.lower() != EXPECTED_SHA256.lower():
    raise RuntimeError("Source SHA-256 mismatch. Stop and investigate.")

# Load strings as labels when needed; continuous fields must remain numeric.
df = pd.read_csv(SOURCE)
if df.shape != EXPECTED_SHAPE or df.columns.tolist() != EXPECTED_HEADER:
    raise RuntimeError(f"Source shape/header mismatch: {df.shape}. Stop and investigate.")

print("Source identity: PASS")
print(f"File size: {source_size:,} bytes | Shape: {df.shape}")
print(f"SHA-256: {source_hash}")


Source identity: PASS
File size: 70,025,339 bytes | Shape: (188318, 132)
SHA-256: 74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03


# Confirm the distribution inventory covers all 14 fields

In [5]:
cont_cols = [f'cont{i}' for i in range(1, 15)]
inventory = df[cont_cols].describe().T
inventory['skew'] = df[cont_cols].skew()
inventory.to_csv('cont_distribution_inventory.csv')
print(f"Fields covered: {len(inventory)}")  # confirm this prints 14
inventory

Fields covered: 14


,count,mean,std,min,25%,50%,75%,max,skew
cont1,188318.0,0.493861,0.187640,0.000016,0.346090,0.475784,0.623912,0.984975,0.516424
cont2,188318.0,0.507188,0.207202,0.001149,0.358319,0.555782,0.681761,0.862654,-0.310941
cont3,188318.0,0.498918,0.202105,0.002634,0.336963,0.527991,0.634224,0.944251,-0.010002
cont4,188318.0,0.491812,0.211292,0.176921,0.327354,0.452887,0.652072,0.954297,0.416096
cont5,188318.0,0.487428,0.209027,0.281143,0.281143,0.422268,0.643315,0.983674,0.681622
cont6,188318.0,0.490945,0.205273,0.012683,0.336105,0.440945,0.655021,0.997162,0.461214
cont7,188318.0,0.484970,0.178450,0.069503,0.350175,0.438285,0.591045,1.000000,0.826053
cont8,188318.0,0.486437,0.199370,0.236880,0.312800,0.441060,0.623580,0.980200,0.676634
cont9,188318.0,0.485506,0.181660,0.000080,0.358970,0.441450,0.566820,0.995400,1.072429
cont10,188318.0,0.498066,0.185877,0.000000,0.364580,0.461190,0.614590,0.994980,0.355001


# Reproduce the raw Pearson correlations with loss

In [6]:
corrs_with_loss = df[cont_cols].corrwith(df['loss']).sort_values(key=lambda x: x.abs(), ascending=False)
corrs_with_loss

cont2     0.141528
cont7     0.119799
cont3     0.111053
cont11    0.099806
cont12    0.098724
cont6     0.040967
cont4    -0.035831
cont8     0.030508
cont10    0.020236
cont14    0.019298
cont9     0.014456
cont5    -0.011355
cont1    -0.010237
cont13   -0.004022
dtype: float64

# Explanation

A raw Pearson correlation only measures linear association. None of the 14 continuous fields shows a strong linear relationship with raw loss on their own. The strongest is cont2 at ~0.14, which is a weak linear signal. This does not mean these fields lack predictive value. Pearson correlation can't detect nonlinear relationships, threshold effects, or interactions with other fields.

# Binned summaries to surface nonlinear patterns

In [7]:
def binned_summary(df, field, target='loss', bins=10):
    df_bin = df.copy()
    df_bin[f'{field}_bin'] = pd.qcut(df_bin[field], q=bins, duplicates='drop')
    return df_bin.groupby(f'{field}_bin', observed=True)[target].agg(['mean', 'median', 'std', 'count'])

cont2_binned = binned_summary(df, 'cont2')
cont2_binned.to_csv('cont2_binned_summary.csv')
cont2_binned

,mean,median,std,count
cont2_bin,,,,
"(0.00014999999999999996, 0.2]",2120.267592,1740.260,1503.641588,21257
"(0.2, 0.299]",2650.054737,2109.640,2002.699112,21001
"(0.299, 0.358]",2815.646257,2141.760,2233.165582,14370
"(0.358, 0.489]",3008.030695,2190.185,2547.292437,35670
"(0.489, 0.556]",3216.650567,2221.060,3004.055635,20975
"(0.556, 0.621]",3398.676783,2258.145,3414.056744,20190
"(0.621, 0.737]",3399.239682,2225.165,3411.836984,32172
"(0.737, 0.786]",3422.421956,2107.070,3744.139000,15475
"(0.786, 0.863]",3481.255580,2123.250,3795.258897,7208


# Inter-field correlations

In [8]:
corr_matrix = df[cont_cols].corr()
pairs = []
for i in range(len(cont_cols)):
    for j in range(i+1, len(cont_cols)):
        c1, c2 = cont_cols[i], cont_cols[j]
        pairs.append((c1, c2, corr_matrix.loc[c1, c2]))
pairs_df = pd.DataFrame(pairs, columns=['field1','field2','corr']).sort_values(by='corr', key=lambda x: x.abs(), ascending=False)
pairs_df.to_csv('cont_field_correlations.csv', index=False)
pairs_df.head(10)

,field1,field2,corr
85,cont11,cont12,0.994384
7,cont1,cont9,0.929912
58,cont6,cont10,0.883351
61,cont6,cont13,0.815091
8,cont1,cont10,0.808551
57,cont6,cont9,0.797544
76,cont9,cont10,0.785697
60,cont6,cont12,0.785144
59,cont6,cont11,0.773745
4,cont1,cont6,0.758315


Several continuous fields are highly correlated with each other, like cont11 and cont12 at ~0.99, which is close enough to suggest they may be near duplicates or derived from a shared source. High predictor correlation (multicollinearity) matters mainly for linear model interpretation. It inflates coefficient variance and makes individual coefficients unstable or hard to give meaning to, since the model can't cleanly separate two fields that move together. It does not necessarily hurt predictive accuracy, and tree-based models are largely robust to it.